# Mountain Pass - análisis

In [ ]:
import rasterio

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from rasterio.windows import Window
from tqdm.auto import tqdm

Tomar en cuenta que la longitud de onda esta en um (mircrometros)

In [ ]:
base = Path.cwd()
for zona in (base / "Mountain Pass", base, base.parent):
    if (zona / "data" / "EnMAP").is_dir():
        break
else:
    raise FileNotFoundError("Abrí el notebook desde el proyecto Mountain Pass.")
usgs = zona / "data" / "USGS"
ruta_reflectancia = usgs / "splib07b_Monazite_REE_WS385_crystal_ASDFRb_AREF.txt"
ruta_wavelength = usgs / "splib07b_Wavelengths_ASDFR_0.35-2.5microns_2151ch.txt"

In [ ]:
enmap = zona / "data" / "EnMAP"
escena = "ENMAP01-____L2A-DT0000040453_20230906T190259Z_002_V010506_20260703T224650Z"

def archivo_enmap(sufijo):
    return enmap / f"{escena}-{sufijo}"

xml_file = archivo_enmap("METADATA.XML")

Firma espectral de la monacita (USGS)

In [ ]:
def leer_usgs(ruta):
    datos = np.loadtxt(ruta,skiprows=1)
    return datos

In [ ]:
def leer_firma_usgs(ruta_reflectancia,ruta_wavelength):
    reflectancia = leer_usgs(ruta_reflectancia)
    wavelength = leer_usgs(ruta_wavelength)
    firma = pd.DataFrame({
        "Wavelength_nm": wavelength*1000,
        "Reflectance": reflectancia})

    return firma

In [ ]:
firma = leer_firma_usgs(ruta_reflectancia, ruta_wavelength)
firma.head()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(firma["Wavelength_nm"], firma["Reflectance"])

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")

plt.title("Firma espectral USGS")
plt.grid()
plt.show()

Ver las caracteristicas del EnMap

In [ ]:
def leer_sensor_enmap(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    bandas = root.find(".//bandCharacterisation")
    centros=[]
    fwhm=[]

    for banda in bandas:
        centros.append(float(banda.find("wavelengthCenterOfBand").text))
        fwhm.append(float(banda.find("FWHMOfBand").text))

    sensor = pd.DataFrame({
        "Band":range(1,len(centros)+1),
        "Center_nm":centros,
        "FWHM_nm":fwhm
    })

    return sensor

In [ ]:
sensor_enmap = leer_sensor_enmap(xml_file)
sensor_enmap.head()

Empezando con el Spectral Resampling

In [ ]:
def fwhm_to_sigma(fwhm):
    sigma = fwhm/(2*np.sqrt(2*np.log(2)))
    return sigma

In [ ]:
def gaussian_response(wavelength,center,sigma):
    respuesta = np.exp(-((wavelength-center)**2)/(2*sigma**2))
    return respuesta

In [ ]:
def spectral_resampling(firma,center,fwhm):
    wavelengths = firma["Wavelength_nm"].values
    reflectance = firma["Reflectance"].values
    sigma = fwhm_to_sigma(fwhm)
    respuesta = gaussian_response(wavelengths, center, sigma)
    reflectancia = np.sum(reflectance*respuesta)/np.sum(respuesta)
    return reflectancia

In [ ]:
firma_enmap=[]

for _, banda in sensor_enmap.iterrows():
    r = spectral_resampling(firma, banda["Center_nm"], banda["FWHM_nm"])
    firma_enmap.append(r)

In [ ]:
firma_enmap = pd.DataFrame({
    "Band":sensor_enmap["Band"],
    "Center_nm":sensor_enmap["Center_nm"],
    "Reflectance":firma_enmap})

In [ ]:
firma_enmap.head()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    firma_enmap["Center_nm"],
    firma_enmap["Reflectance"],
    markersize=3)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Firma espectral simulada para EnMAP")
plt.grid()

plt.show()

Leer el cubo de estudio

In [ ]:
ruta_cubo = archivo_enmap("SPECTRAL_IMAGE.TIF")

In [ ]:
with rasterio.open(ruta_cubo) as src:
    alto, ancho = src.shape
    assert src.count == len(sensor_enmap)

def leer_reflectancia(src, bandas=None, ventana=None):
    valores = src.read(bandas, window=ventana, out_dtype="float64")
    valores[valores == -32768] = np.nan
    return valores / 10000.0

def leer_pixel(fila, columna, bandas=None):
    with rasterio.open(ruta_cubo) as src:
        return leer_reflectancia(src, bandas, Window(int(columna), int(fila), 1, 1))[:, 0, 0]

Mascara de calidad

In [ ]:
ruta_cloud = archivo_enmap("QL_QUALITY_CLOUD.TIF")
ruta_shadow = archivo_enmap("QL_QUALITY_CLOUDSHADOW.TIF")
ruta_cirrus = archivo_enmap("QL_QUALITY_CIRRUS.TIF")
ruta_haze = archivo_enmap("QL_QUALITY_HAZE.TIF")
ruta_snow = archivo_enmap("QL_QUALITY_SNOW.TIF")
ruta_pixelmask = archivo_enmap("QL_PIXELMASK.TIF")

In [ ]:
import rasterio

def leer_mascara(ruta):

    with rasterio.open(ruta) as src:

        return src.read(1)

In [ ]:
cloud = leer_mascara(ruta_cloud)

shadow = leer_mascara(ruta_shadow)

cirrus = leer_mascara(ruta_cirrus)

haze = leer_mascara(ruta_haze)

snow = leer_mascara(ruta_snow)

pixelmask = leer_mascara(ruta_pixelmask)

In [ ]:
mascara_calidad = (
    (cloud == 1) |
    (shadow == 1) |
    (cirrus > 0) |
    (haze == 1) |
    (pixelmask == 1))

print("Pixeles descartados:",
      np.sum(mascara_calidad))

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(mascara_calidad, cmap="gray")

plt.title("Máscara de calidad")
plt.axis("off")

plt.show()

```

SERGIOS APPLIES A FILTER MASK

SAM

In [ ]:
firma_referencia = firma_enmap["Reflectance"].values

In [ ]:
# Define your clean, usable wavelength windows (in nm)
wavelengths = sensor_enmap["Center_nm"].values

# Create a boolean mask of bands to KEEP
good_bands_mask = (
    (wavelengths < 890) | # Keep VNIR before the noisy overlap region
    ((wavelengths > 1000) & (wavelengths < 1340)) | # Keep SWIR before 1st water band
    ((wavelengths > 1460) & (wavelengths < 1780)) | # Keep SWIR between water bands
    ((wavelengths > 1960) & (wavelengths < 2400))   # Keep SWIR after 2nd water band
)

# Apply the filter to your reference spectrum
filtered_firma_referencia = firma_referencia[good_bands_mask]
filtered_centers = wavelengths[good_bands_mask]

bandas = sensor_enmap.loc[good_bands_mask, "Band"].tolist()

In [ ]:
def spectral_angle_mapper(firma1, firma2):

    mascara = (~np.isnan(firma1)) & (~np.isnan(firma2))

    firma1 = firma1[mascara]
    firma2 = firma2[mascara]

    # Si no quedan bandas válidas
    if len(firma1) == 0:
        return np.nan

    norma1 = np.linalg.norm(firma1)
    norma2 = np.linalg.norm(firma2)

    # Evitar división por cero
    if norma1 == 0 or norma2 == 0:
        return np.nan

    cos_theta = np.dot(firma1, firma2) / (norma1 * norma2)

    cos_theta = np.clip(cos_theta, -1, 1)

    theta = np.arccos(cos_theta)

    return theta

In [ ]:
def mapa_sam(bandas, referencia):
    mapa = np.full((alto, ancho), np.nan)
    conteo = np.zeros((alto, ancho), dtype=np.uint16)
    ref = referencia[:, None, None]
    with rasterio.Env(GDAL_CACHEMAX=128 * 1024**2), rasterio.open(ruta_cubo) as src:
        for fila in tqdm(range(0, alto, 16), desc="SAM por bloques"):
            filas = slice(fila, min(fila + 16, alto))
            ventana = Window(0, fila, ancho, filas.stop - fila)
            bloque = leer_reflectancia(src, bandas, ventana)
            validos = np.isfinite(bloque) & np.isfinite(ref)
            conteo[filas] = validos.sum(axis=0)
            valores = np.where(validos, bloque, 0)
            producto = np.sum(valores * np.where(np.isfinite(ref), ref, 0), axis=0)
            norma = np.sqrt(np.sum(valores**2, axis=0) * np.sum(np.where(validos, ref**2, 0), axis=0))
            usar = (norma > 0) & ~mascara_calidad[filas]
            coseno = np.full(norma.shape, np.nan)
            np.divide(producto, norma, out=coseno, where=usar)
            mapa[filas] = np.arccos(np.clip(coseno, -1, 1))
    return mapa, conteo

In [ ]:
sam_map, bandas_validas = mapa_sam(bandas, filtered_firma_referencia)

In [ ]:
plt.figure(figsize=(12,10))

plt.imshow(
    sam_map,
    cmap="hot",
    vmin=np.nanpercentile(sam_map,2),
    vmax=np.nanpercentile(sam_map,98)
)

plt.colorbar(label="Ángulo SAM (radianes)")

plt.title("Mapa de similitud espectral SAM - Monacita")

plt.xlabel("Columnas")

plt.ylabel("Filas")

plt.show()

In [ ]:
angulos = sam_map.flatten()

In [ ]:
angulos = sam_map.flatten()

# Eliminar NaN
angulos = angulos[~np.isnan(angulos)]

# Ordenarlos
angulos = np.sort(angulos)

print(angulos[:5])

comprobacion

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    sam_map[~np.isnan(sam_map)],
    bins=100,
    color="steelblue"
)

plt.xlabel("Angulo SAM")
plt.ylabel("Numero de pixeles")
plt.title("Distribucion de angulos SAM")
plt.grid(True)

plt.show()

In [ ]:
print(np.unique(cloud))
print(np.unique(shadow))
print(np.unique(cirrus))
print(np.unique(haze))
print(np.unique(snow))
print(np.unique(pixelmask))

In [ ]:
umbral = 0.17

mascara = sam_map < umbral


In [ ]:
print(np.sum(mascara))

In [ ]:
# Obtener las coordenadas de los píxeles dentro del umbral
filas, columnas = np.where(mascara)

print("Pixeles dentro del umbral:", len(filas))

plt.figure(figsize=(12, 6))

# Firma de referencia de monazita
plt.plot(filtered_centers, filtered_firma_referencia, color="black", linewidth=3, label="Monazita")

# Graficar los pixeles dentro del umbral
for i, (fila, col) in enumerate(zip(filas, columnas), start=1):

    firma_pixel = leer_pixel(fila, col, bandas)

    angulo = sam_map[fila, col]

    plt.plot(filtered_centers, firma_pixel, linewidth=1.5, alpha=0.7, label=f"P{i} ({fila},{col}) = {angulo:.4f} rad")

# Absorciones del Nd
for wl in [580, 745, 801, 872, 1235, 1469]:
    plt.axvline(wl, color="gray", linestyle="--", alpha=0.5)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title(f"Pixeles con SAM < {umbral} rad vs Monazita")

# plt.xlim(500, 900)

plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.show()

In [ ]:
nd_mask = (
    (filtered_centers >= 500) &
    (filtered_centers <= 900)
)

# Aplicar la mascara a la firma de referencia
firma_nd = filtered_firma_referencia[nd_mask]

# Aplicar la mascara al cubo
bandas_nd = np.asarray(bandas)[nd_mask].tolist()

# Centros de longitud de onda correspondientes
centers_nd = filtered_centers[nd_mask]

print("Número de bandas usadas:", len(centers_nd))
print("Rango:", centers_nd.min(), "–", centers_nd.max(), "nm")

In [ ]:
sam_map_nd, _ = mapa_sam(bandas_nd, firma_nd)

In [ ]:
umbral_nd = 0.17

mascara_nd = sam_map_nd < umbral_nd

print("Píxeles dentro del umbral:", np.sum(mascara_nd))

In [ ]:
filas_nd, columnas_nd = np.where(mascara_nd)

print("Píxeles encontrados:", len(filas_nd))

for i, (f, c) in enumerate(
    zip(filas_nd, columnas_nd),
    start=1
):

    angulo = sam_map_nd[f, c]

    print(
        f"Píxel {i}: "
        f"fila={f}, columna={c}, "
        f"ángulo={angulo:.6f} rad"
    )

In [ ]:
plt.figure(figsize=(12, 6))

# Firma de monazita en 500–900 nm
plt.plot(
    centers_nd,
    firma_nd,
    color="black",
    linewidth=3,
    label="Monazita"
)
print("Pixeles dentro del umbral:", len(filas))

# Todos los píxeles que cumplen el umbral
for i, (fila, col) in enumerate(
    zip(filas_nd, columnas_nd),
    start=1
):

    firma_pixel_nd = leer_pixel(fila, col, bandas_nd)

    angulo = sam_map_nd[fila, col]

    plt.plot(
        centers_nd,
        firma_pixel_nd,
        linewidth=1.5,
        alpha=0.7,
        label=f"P{i} ({fila},{col}) = {angulo:.4f} rad"
    )

# Absorciones que estamos estudiando
for wl in [580, 746, 801, 872]:

    plt.axvline(
        wl,
        color="gray",
        linestyle="--",
        alpha=0.5
    )

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")

plt.title(
    f"Pixeles con SAM 500–900 nm < {umbral_nd}"
)

plt.grid(True)

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,10))

plt.imshow(
    mascara,
    cmap="gray"
)

plt.title("Posibles píxeles de monacita")

plt.axis("off")

plt.show()

In [ ]:
with rasterio.open(ruta_cubo) as src:
    red, green, blue = leer_reflectancia(src, [43, 28, 10])

In [ ]:
red[red == -32768] = np.nan
green[green == -32768] = np.nan
blue[blue == -32768] = np.nan

In [ ]:
def normalize(band):
    p2 = np.nanpercentile(band, 2)
    p98 = np.nanpercentile(band, 98)

    band = np.clip(band, p2, p98)
    band = (band - p2) / (p98 - p2)

    return band

In [ ]:
rgb = np.dstack([
    normalize(red),
    normalize(green),
    normalize(blue)
])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
plt.imshow(rgb)
plt.title("Mountain Pass (EnMap)")
plt.axis('off')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rgb_sam = rgb.copy()

plt.figure(figsize=(12,12))
plt.imshow(rgb_sam)

filas, columnas = np.where(mascara)

for f, c in zip(filas, columnas):
    plt.annotate(
        "",
        xy=(c, f),
        xytext=(c+25, f-25),
        arrowprops=dict(
            arrowstyle="->",
            color="red",
            lw=2)
    )

plt.title("Posibles ocurrencias de monacita (SAM)")
plt.axis("off")
plt.show()

In [ ]:
filas, columnas = np.where(mascara)

plt.figure(figsize=(12,12))
plt.imshow(rgb)

for i, (f, c) in enumerate(zip(filas, columnas), start=1):

    plt.annotate(
        str(i),
        xy=(c, f),
        xytext=(c+20, f-20),
        color="red",
        fontsize=12,
        fontweight="bold",
        arrowprops=dict(
            arrowstyle="->",
            color="red",
            lw=2
        )
    )

plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt



plt.figure(figsize=(12, 12))

# Mostrar imagen RGB
plt.imshow(rgb)

# Recorrer los mismos píxeles usados en la gráfica espectral
for i, (fila, col) in enumerate(
    zip(filas_nd, columnas_nd),
    start=1
):

    plt.annotate(
        f"{i}",
        xy=(col, fila),
        xytext=(col + 20, fila - 20),

        color="red",
        fontsize=12,
        fontweight="bold",

        arrowprops=dict(
            arrowstyle="->",
            color="red",
            lw=2
        )
    )

# Título
plt.title(
    f"Ubicacion de los pixeles (500 nm - 900 nm) con SAM < {umbral_nd}"
)

plt.axis("off")

plt.show()

Comparando con el mejor pixel con la monacita

In [ ]:
print(np.sum(mascara))

In [ ]:
print(np.sum(mascara))

for i, (f, c) in enumerate(zip(filas, columnas), start=1):
    print(f"Píxel {i}: fila={f}, columna={c}")

filas, columnas = np.where(mascara)

fila = 142
col = 684

# Extraer la firma del pixel directamente desde el cubo filtrado
firma_pixel_filtrada = leer_pixel(fila, col, bandas)

plt.figure(figsize=(12,6))

# Graficar la referencia de Monacita filtrada
plt.plot(filtered_centers, filtered_firma_referencia, color="black", linewidth=2.5, label="Monacita (Filtrada)")

# Graficar el pixel de Mountain Pass filtrado
plt.plot(filtered_centers, firma_pixel_filtrada, color="red", linewidth=2, label=f"Pixel ({fila}, {col}) Filtrado")

# Bandas del neodimio visibles en el rango limpio
for wl in [580, 746, 801, 872, 1235]:
    plt.axvline(wl, color="gray", linestyle="--", alpha=0.5)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Monacita vs Pixel de Mountain Pass (Zonas Limpias)")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
angulo = spectral_angle_mapper(firma_pixel_filtrada, filtered_firma_referencia)
print(f"Angulo = {angulo:.6f} rad")
print(f"Angulo = {np.degrees(angulo):.2f}°")

In [ ]:
import pysptools.spectro as spectro

In [ ]:
espectro = filtered_firma_referencia.tolist()
longitudes = filtered_centers.tolist()

schq_ref = spectro.SpectrumConvexHullQuotient(
    espectro,
    longitudes
)

firma_ref_cr = schq_ref.get_continuum_removed_spectrum()

In [ ]:
espectro_pixel = firma_pixel_filtrada.tolist()

schq_pixel = spectro.SpectrumConvexHullQuotient(
    espectro_pixel,
    longitudes
)

firma_pixel_cr = schq_pixel.get_continuum_removed_spectrum()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(filtered_centers,
         firma_ref_cr,
         label="Monacita",
         linewidth=2)

plt.plot(filtered_centers,
         firma_pixel_cr,
         label="Píxel",
         color="red",
         linewidth=2)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Continuum Removal")
plt.title("Monacita vs Píxel")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,12))
plt.imshow(rgb)

plt.annotate(
    f"({fila}, {col})",
    xy=(col, fila),
    xytext=(col+40, fila-40),
    color="red",
    fontsize=12,
    arrowprops=dict(
        arrowstyle="->",
        color="red",
        lw=2
    )
)

plt.title("Ubicacion del pixel")
plt.axis("off")
plt.show()

Ahora utilizando una libreria en vez la gausiana

In [ ]:
from spectres import spectres

In [ ]:
reflectancia_spectres = spectres(
    sensor_enmap["Center_nm"].values,
    firma["Wavelength_nm"].values,
    firma["Reflectance"].values
)

In [ ]:
firma_spectres = sensor_enmap.copy()

firma_spectres["Reflectance"] = reflectancia_spectres

firma_spectres.head()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    firma_enmap["Center_nm"],
    firma_enmap["Reflectance"],
    label="El Gaussiano",
    linewidth=2
)

plt.plot(
    firma_spectres["Center_nm"],
    firma_spectres["Reflectance"],
    "--",
    label="SpectRes",
    linewidth=2
)

plt.plot(firma["Wavelength_nm"], firma["Reflectance"], "--", label="La og")


plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Comparacion del remuestreo espectral")
plt.grid(True)
plt.legend()

plt.show()

Empezando con el continum

Ahora Aqui una prueba

In [ ]:
fila = 150
col = 691

firma_prueba = leer_pixel(fila, col, bandas)

In [ ]:
mask_vnir = ((filtered_centers >= 520) & (filtered_centers <= 900))
wavelength_vnir = filtered_centers[mask_vnir]
reflectance_vnir = firma_prueba[mask_vnir]

In [ ]:
import pysptools.spectro as spectro

# Firma del pixel
espectro_pixel = reflectance_vnir.tolist()

# Longitudes de onda correspondientes
longitudes = wavelength_vnir.tolist()

# Continuum removal mediante convex hull
schq_pixel = spectro.SpectrumConvexHullQuotient(espectro_pixel, longitudes)

# Espectro continuum-removed
firma_pixel_cr = np.array(schq_pixel.get_continuum_removed_spectrum())

In [ ]:
derivada_cr = np.gradient(firma_pixel_cr, wavelength_vnir)

VENTANAS

In [ ]:
# IDs de las bandas después del filtro espectral general
filtered_bands = sensor_enmap["Band"].values[good_bands_mask]

# IDs correspondientes al VNIR 520–900 nm
band_ids_vnir = filtered_bands[mask_vnir]

In [ ]:
mask_800 = ((wavelength_vnir >= 770) & (wavelength_vnir <= 825))

wl_800 = wavelength_vnir[mask_800]
cr_800 = firma_pixel_cr[mask_800]

In [ ]:
mask_580 = ((wavelength_vnir >= 565) & (wavelength_vnir <= 605))

wl_580 = wavelength_vnir[mask_580]
cr_580 = firma_pixel_cr[mask_580]

In [ ]:
mask_740 = ((wavelength_vnir >= 720) & (wavelength_vnir <= 778) & (band_ids_vnir != 62))

wl_740 = wavelength_vnir[mask_740]
cr_740 = firma_pixel_cr[mask_740]

In [ ]:
mask_870 = ((wavelength_vnir >= 825) & (wavelength_vnir <= 895))

wl_870 = wavelength_vnir[mask_870]
cr_870 = firma_pixel_cr[mask_870]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

centros = firma_enmap["Center_nm"].to_numpy()
reflectancia = firma_enmap["Reflectance"].to_numpy()

separacion = np.diff(centros)

print("Separación máxima entre bandas:",
      np.max(separacion), "nm")

print("\nSaltos espectrales mayores a 20 nm:")

for i in np.where(separacion > 20)[0]:
    print(
        f"{centros[i]:.2f} nm → "
        f"{centros[i+1]:.2f} nm | "
        f"Separación: {separacion[i]:.2f} nm"
    )

plt.figure(figsize=(13, 5))

plt.plot(
    firma["Wavelength_nm"],
    firma["Reflectance"],
    color="green",
    alpha=0.6,
    label="USGS original"
)

plt.scatter(
    centros,
    reflectancia,
    color="blue",
    s=12,
    label="Gaussiano: bandas EnMAP"
)

plt.xlim(1700, 2050)
plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Inspección de bandas EnMAP sin conectar puntos")
plt.grid()
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

centros = firma_enmap["Center_nm"].to_numpy()
valores = firma_enmap["Reflectance"].to_numpy()

orden = np.argsort(centros)
centros = centros[orden]
valores = valores[orden]

# Solo para visualización: separar tramos con saltos > 20 nm
x_graf = []
y_graf = []

for i in range(len(centros)):
    if i > 0 and centros[i] - centros[i-1] > 20:
        x_graf.append(np.nan)
        y_graf.append(np.nan)

    x_graf.append(centros[i])
    y_graf.append(valores[i])

plt.figure(figsize=(13, 5))

plt.plot(
    firma["Wavelength_nm"],
    firma["Reflectance"],
    "--",
    color="green",
    alpha=0.6,
    label="USGS original"
)

plt.plot(
    x_graf,
    y_graf,
    "o-",
    markersize=3,
    color="blue",
    label="Gaussiano EnMAP"
)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Firma USGS y remuestreo EnMAP sin conexiones artificiales")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
print("USGS original:")
print("Mínimo:", firma["Wavelength_nm"].min())
print("Máximo:", firma["Wavelength_nm"].max())

print("\nFirma remuestreada EnMAP:")
print("Mínimo:", firma_enmap["Center_nm"].min())
print("Máximo:", firma_enmap["Center_nm"].max())

print("\nPrimeras 15 bandas EnMAP:")
print(firma_enmap[["Center_nm", "Reflectance"]].head(15))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

# Firma original de laboratorio
plt.plot(
    firma["Wavelength_nm"],
    firma["Reflectance"],
    color="green",
    linewidth=2,
    label="USGS original"
)

# Firma remuestreada
plt.plot(
    firma_enmap["Center_nm"],
    firma_enmap["Reflectance"],
    "o-",
    color="blue",
    markersize=4,
    label="Remuestreo gaussiano EnMAP"
)

# Regiones diagnósticas aproximadas de Nd
plt.axvline(
    740,
    color="red",
    linestyle="--",
    alpha=0.6,
    label="Nd ~740 nm"
)

plt.axvline(
    800,
    color="orange",
    linestyle="--",
    alpha=0.6,
    label="Nd ~800 nm"
)

plt.xlim(700, 850)
plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Comparación espectral en la región diagnóstica del Nd")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Inspeccionar las bandas EnMAP próximas a 1469 nm

region = firma_enmap[
    (firma_enmap["Center_nm"] >= 1420) &
    (firma_enmap["Center_nm"] <= 1520)
]

print(region[["Center_nm", "Reflectance"]].to_string(index=False))

In [ ]:
import numpy as np

wavelengths = sensor_enmap["Center_nm"].to_numpy()

good_bands_mask = (
    (wavelengths < 890) |
    ((wavelengths > 1000) & (wavelengths < 1340)) |
    ((wavelengths > 1460) & (wavelengths < 1780)) |
    ((wavelengths > 1960) & (wavelengths < 2400))
)

excluded_bands = sensor_enmap.loc[~good_bands_mask]

print("Bandas totales:", len(sensor_enmap))
print("Bandas conservadas:", good_bands_mask.sum())
print("Bandas excluidas:", (~good_bands_mask).sum())

print("\nBandas excluidas:")
print(excluded_bands[["Center_nm", "FWHM_nm"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    firma_enmap["Center_nm"],
    firma_enmap["Reflectance"],
    "o-",
    markersize=3,
    label="USGS remuestreada"
)

plt.axvspan(
    890, 1000,
    color="red",
    alpha=0.15,
    label="Región excluida por la máscara"
)

plt.xlim(850, 1050)
plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Inspección espectral de la región 890–1000 nm")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Centros de banda EnMAP
wavelengths = sensor_enmap["Center_nm"].to_numpy()

# Elegir un píxel aproximadamente central
fila = alto // 2
columna = ancho // 2

# Extraer su firma espectral
espectro = leer_pixel(fila, columna)

# Seleccionar región de interés
region = (wavelengths >= 850) & (wavelengths <= 1050)

plt.figure(figsize=(12, 5))

plt.plot(
    wavelengths[region],
    espectro[region],
    "o-",
    markersize=4,
    label="Píxel real EnMAP"
)

plt.axvspan(
    890, 1000,
    color="red",
    alpha=0.15,
    label="Región excluida"
)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Inspección de un píxel EnMAP: 850–1050 nm")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wavelengths = sensor_enmap["Center_nm"].to_numpy()

region = (wavelengths >= 850) & (wavelengths <= 1050)

# Seleccionar cinco posiciones de la imagen


posiciones = [
    (alto // 4, ancho // 4),
    (alto // 4, 3 * ancho // 4),
    (alto // 2, ancho // 2),
    (3 * alto // 4, ancho // 4),
    (3 * alto // 4, 3 * ancho // 4)
]

plt.figure(figsize=(13, 6))

for fila, columna in posiciones:
    espectro = leer_pixel(fila, columna)

    if np.all(np.isfinite(espectro[region])):
        plt.plot(
            wavelengths[region],
            espectro[region],
            "o-",
            markersize=2,
            alpha=0.75,
            label=f"Píxel ({fila}, {columna})"
        )

plt.axvspan(
    890, 1000,
    color="red",
    alpha=0.12,
    label="Región excluida"
)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia")
plt.title("Comparación de cinco píxeles EnMAP")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Considerar únicamente píxeles que pasan la máscara de calidad
validas = bandas_validas[~mascara_calidad]

print("Bandas seleccionadas:", len(bandas))
print("Mínimo de bandas válidas:", validas.min())
print("Máximo de bandas válidas:", validas.max())
print("Promedio de bandas válidas:", validas.mean())

print("Píxeles con todas las bandas válidas:",
      np.sum(validas == len(bandas)))

print("Píxeles con menos de 150 bandas válidas:",
      np.sum(validas < 150))

In [ ]:
validas = bandas_validas[~mascara_calidad]

print("Distribución de bandas válidas:")

for minimo, maximo in [
    (0, 0),
    (1, 49),
    (50, 99),
    (100, 149),
    (150, 186),
    (187, 187)
]:
    cantidad = np.sum(
        (validas >= minimo) &
        (validas <= maximo)
    )

    print(f"{minimo:3d} - {maximo:3d}: {cantidad:,} píxeles")

In [ ]:
import sys

print(sys.executable)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import tqdm

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Rasterio:", rasterio.__version__)
print("Entorno local funcionando correctamente")